In [1]:
# 04_run_ctdgan.ipynb
# Multi-seed run of the ctdGAN baseline (artsyn / lakritidis-ARTSyn).
# ctdGAN is internally seeded and does joint class+GMM-cluster conditioning on the
# FULL training set; it is the slowest baseline. On the 3 largest datasets
# (fraud_detection, unsw_nb15, protein_homo) a single seed already costs hours, so
# those are left to a single-seed run (see README); here we cover the other 13.
# Output: ../results/tables/raw_ctdgan.csv   ENV: artsyn==0.5.2, scikit-learn, lightgbm, torch.
# ---- Shared configuration and protocol (identical across run notebooks) ----
# This block is self-contained so each notebook runs independently in its own
# conda environment. All 16 datasets must sit in ../data as <name>.csv with a
# 'target' column (1 = minority/positive).
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             average_precision_score, balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

DATA_DIR   = "../data"
TABLES_DIR = "../results/tables"
os.makedirs(TABLES_DIR, exist_ok=True)

# Evaluation protocol for every method: 5 repeated stratified 70/30 holdouts.
SEEDS = [42, 43, 44, 45, 46]
# Optional: set to a short list (e.g. ["ecoli", "pima_diabetes"]) to test quickly.
DATASETS_SUBSET = None

DATASETS = ["credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
            "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
            "pageblocks", "protein_homo", "satellite", "churn", "secom",
            "thyroid_sick", "unsw_nb15"]
CLASSIFIERS = ("RF", "LGBM", "MLP")
RAW_COLUMNS = ["dataset", "method", "classifier", "seed", "AUC", "PR_AUC", "F1",
               "Precision", "Recall", "G_mean", "Balanced_Acc",
               "generation_time", "train_time", "method_group"]

def load_dataset(name):
    """Read ../data/<name>.csv -> (X float64 ndarray, y int ndarray)."""
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))
    bcols = df.select_dtypes(include="bool").columns
    if len(bcols):
        df[bcols] = df[bcols].astype("float64")
    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)
    return X, y

def get_classifier(name, variant, random_state):
    """variant: 'generative' (notebooks 03/04x config) or 'classic' (notebook 02)."""
    if name == "RF":
        return RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=3,
                                      class_weight="balanced", random_state=random_state, n_jobs=-1)
    if name == "LGBM":
        from lightgbm import LGBMClassifier
        if variant == "classic":
            return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                                  random_state=random_state, verbosity=-1, n_jobs=-1)
        return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                              class_weight="balanced", random_state=random_state, n_jobs=-1, verbose=-1)
    if name == "MLP":
        if variant == "classic":
            return MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                                 alpha=0.001, random_state=random_state, max_iter=500)
        return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001,
                             max_iter=300, random_state=random_state)
    raise ValueError(name)

def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sens * spec))

def evaluate(model, X_test, y_test):
    yp = model.predict_proba(X_test)[:, 1]
    yd = model.predict(X_test)
    return {"AUC": round(roc_auc_score(y_test, yp), 4),
            "PR_AUC": round(average_precision_score(y_test, yp), 4),
            "F1": round(f1_score(y_test, yd, average="macro", zero_division=0), 4),
            "Precision": round(precision_score(y_test, yd, average="macro", zero_division=0), 4),
            "Recall": round(recall_score(y_test, yd, average="macro", zero_division=0), 4),
            "G_mean": round(g_mean_score(y_test, yd), 4),
            "Balanced_Acc": round(balanced_accuracy_score(y_test, yd), 4)}

def holdout_split_scale(X, y, seed):
    """Single stratified 70/30 holdout; StandardScaler fit on train only."""
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    sc = StandardScaler()
    return sc.fit_transform(Xtr), sc.transform(Xte), ytr, yte

def append_rows(out_csv, rows):
    pd.DataFrame(rows, columns=RAW_COLUMNS).to_csv(
        out_csv, mode="a", header=not os.path.exists(out_csv), index=False)

def seed_everything(seed):
    np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed)
    except Exception:
        pass

def run_oversampler(method, method_group, oversample_fn, out_csv, variant="generative",
                    skip_datasets=()):
    """Generic driver: for each seed x dataset, oversample then fit/evaluate 3 classifiers.
    oversample_fn(X_tr_scaled, y_tr, seed) -> (X_res, y_res, generation_time_or_None)."""
    datasets = DATASETS_SUBSET or DATASETS
    for seed in SEEDS:
        for ds in datasets:
            if ds in skip_datasets:
                continue
            try:
                X, y = load_dataset(ds)
            except FileNotFoundError:
                print(f"[missing csv] {ds}"); continue
            Xtr, Xte, ytr, yte = holdout_split_scale(X, y, seed)
            seed_everything(seed)
            t0 = time.time()
            Xr, yr, gt = oversample_fn(Xtr, ytr, seed)
            gen_time = round(time.time() - t0, 3) if gt is None else round(gt, 3)
            rows = []
            for cn in CLASSIFIERS:
                clf = get_classifier(cn, variant, random_state=seed)
                t1 = time.time(); clf.fit(Xr, yr); tt = round(time.time() - t1, 3)
                rows.append({"dataset": ds, "method": method, "classifier": cn, "seed": seed,
                             **evaluate(clf, Xte, yte), "generation_time": gen_time,
                             "train_time": tt, "method_group": method_group})
            append_rows(out_csv, rows)
            print(f"[ok] {method:12s} seed={seed} {ds:16s} "
                  f"AUC(RF/LGBM/MLP)={rows[0]['AUC']}/{rows[1]['AUC']}/{rows[2]['AUC']}")

from artsyn.generators.ctd_gan import ctdGAN
CTDGAN_EPOCHS = 100
OUT = os.path.join(TABLES_DIR, "raw_ctdgan.csv")

# For the large datasets, keep a single seed (set to [42]); edit as needed.
LARGE = {"fraud_detection", "unsw_nb15", "protein_homo"}

def apply_ctdgan(Xtr, ytr, seed):
    m = ctdGAN(epochs=CTDGAN_EPOCHS, random_state=seed, cluster_method="gmm",
               sampling_strategy="auto")
    Xr, yr = m.fit_resample(Xtr, ytr)
    return Xr, yr, None

datasets = DATASETS_SUBSET or DATASETS
for seed in SEEDS:
    for ds in datasets:
        if ds in LARGE and seed != SEEDS[0]:
            continue  # single seed only for the 3 largest datasets
        try:
            X, y = load_dataset(ds)
        except FileNotFoundError:
            print(f"[missing csv] {ds}"); continue
        Xtr, Xte, ytr, yte = holdout_split_scale(X, y, seed)
        seed_everything(seed)
        t0 = time.time(); Xr, yr, _ = apply_ctdgan(Xtr, ytr, seed); gt = round(time.time() - t0, 3)
        rows = []
        for cn in CLASSIFIERS:
            clf = get_classifier(cn, "generative", random_state=seed)
            t1 = time.time(); clf.fit(Xr, yr); tt = round(time.time() - t1, 3)
            rows.append({"dataset": ds, "method": "ctdGAN", "classifier": cn, "seed": seed,
                         **evaluate(clf, Xte, yte), "generation_time": gt,
                         "train_time": tt, "method_group": "new_generative"})
        append_rows(OUT, rows)
        print(f"[ok] ctdGAN seed={seed} {ds:16s} gen={gt}s "
              f"AUC={rows[0]['AUC']}/{rows[1]['AUC']}/{rows[2]['AUC']}")
print("DONE -> ", OUT)


ctdGAN Sampling     : 100%|██████████| 2/2 [00:01<00:00,  1.68it/s]


[ok] ctdGAN seed=42 credit_default   gen=781.994s AUC=0.7538/0.7762/0.7002


ctdGAN Sampling     : 100%|██████████| 2/2 [00:16<00:00,  8.31s/it]


[ok] ctdGAN seed=42 fraud_detection  gen=7795.146s AUC=0.9563/0.9452/0.9481


ctdGAN Sampling     : 100%|██████████| 2/2 [00:00<00:00, 39.22it/s]


...

[ok] ctdGAN seed=46 churn            gen=117.448s AUC=0.9319/0.9276/0.898


ctdGAN Sampling     : 100%|██████████| 2/2 [00:00<00:00,  5.85it/s]


[ok] ctdGAN seed=46 secom            gen=243.076s AUC=0.671/0.7032/0.6812


ctdGAN Sampling     : 100%|██████████| 2/2 [00:00<00:00,  5.83it/s]


[ok] ctdGAN seed=46 thyroid_sick     gen=117.542s AUC=0.9942/0.9964/0.9623
DONE ->  ../results/tables\raw_ctdgan.csv
